# Measuring a Classifier Honestly: Precision, Recall, Thresholds and Calibration

The runnable code of [Measuring a Classifier Honestly: Precision, Recall, Thresholds and Calibration](https://residual-stream.lvtan71.workers.dev/posts/measuring-a-classifier/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/measuring-a-classifier.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "measuring-a-classifier"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/features-overfitting-regularization/overfit.py', 'shared/features-overfitting-regularization/router_data.py', 'shared/measuring-a-classifier/router_models.py', 'data/router-questions.csv']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## All three routers, compared properly

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/measuring-a-classifier/#all-three-routers-compared-properly)

In [ ]:
%%writefile router_models.py
import numpy as np
from router_data import split, featurize, tokens, vietnamese
from overfit import sigmoid, log_loss, train

SERVING = {"kv", "cache", "caching", "batch", "batching", "latency",
           "throughput", "serve", "serving", "vllm", "sglang", "tensorrt",
           "gpu", "decode", "prefill", "paged", "quantization", "memory"}

def keyword_count(rs):
    return np.array([[sum(t in SERVING for t in tokens(r["question"]))]
                     for r in rs], dtype=float)

def fit_all(seed):
    """Train the three routers of "Features and Overfitting" on one split; return logits."""
    tr, va, te = split(seed)
    ((Xtr, ytr), (Xva, yva), (Xte, yte), (Xvi, yvi)), vocab = featurize(
        tr, va, te, vietnamese)
    kw = [keyword_count(rs) for rs in (tr, va, te, vietnamese)]
    designs = {
        "no penalty": ([Xtr, Xva, Xte, Xvi], 0.0),
        "L2": ([Xtr, Xva, Xte, Xvi], 0.003),
        "L2 + keywords": ([np.c_[X, k] for X, k in zip([Xtr, Xva, Xte, Xvi], kw)], 0.003),
    }
    out = {}
    for name, (mats, lam) in designs.items():
        w, b = train(mats[0], ytr, lam)
        out[name] = {part: (mats[i] @ w + b, y) for i, (part, y) in
                     enumerate([("train", ytr), ("val", yva), ("test", yte), ("vi", yvi)])}
    return out

RUNS = [fit_all(seed) for seed in range(20)]

def pooled(name, part):
    """Logits and labels for one model and one part, pooled over 20 splits."""
    z = np.concatenate([run[name][part][0] for run in RUNS])
    y = np.concatenate([run[name][part][1] for run in RUNS])
    return z, y

In [ ]:
%%writefile measure.py
import numpy as np
from overfit import sigmoid
from router_models import RUNS, pooled

def counts(p, y, t=0.5):
    yes = p >= t
    return ((yes & (y == 1)).sum(), (yes & (y == 0)).sum(),    # TP, FP
            (~yes & (y == 1)).sum(), (~yes & (y == 0)).sum())  # FN, TN

def roc_auc(p, y):
    """Probability that a random positive scores above a random negative."""
    ranks = np.empty(len(p)); ranks[np.argsort(p)] = np.arange(1, len(p) + 1)
    pos, neg = y.sum(), len(y) - y.sum()
    return (ranks[y == 1].sum() - pos * (pos + 1) / 2) / (pos * neg)

def average_precision(p, y):
    """Area under the precision-recall curve (mean precision at each hit)."""
    hits = y[np.argsort(-p)]
    precision = np.cumsum(hits) / np.arange(1, len(hits) + 1)
    return (precision * hits).sum() / hits.sum()

# One split, one model, the default threshold.
z, y = RUNS[0]["L2 + keywords"]["test"]
tp, fp, fn, tn = counts(sigmoid(z), y)
print(f"split 0 test: TP {tp}  FP {fp}  FN {fn}  TN {tn}")

# All three models, 800 test predictions pooled over 20 splits.
for name in ["no penalty", "L2", "L2 + keywords"]:
    z, y = pooled(name, "test")
    p = sigmoid(z)
    tp, fp, fn, tn = counts(p, y)
    precision, recall = tp / (tp + fp), tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    print(f"{name:14s} acc {(tp + tn) / len(y):.3f}  precision {precision:.3f}"
          f"  recall {recall:.3f}  F1 {f1:.3f}  ROC-AUC {roc_auc(p, y):.3f}"
          f"  AP {average_precision(p, y):.3f}")

# The best router's precision-recall trade-off, threshold by threshold.
z, y = pooled("L2 + keywords", "test")
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    tp, fp, fn, tn = counts(sigmoid(z), y, t)
    print(f"threshold {t:.1f}: precision {tp / (tp + fp):.3f}  recall {tp / (tp + fn):.3f}"
          f"  cost {2 * fn + fp}")

# Three ways to pick the threshold when a miss costs twice a false alarm,
# each scored on the test predictions.
grid = np.round(np.arange(0.02, 0.985, 0.01), 2)
cost = lambda p, y, t: 2 * ((p < t) & (y == 1)).sum() + ((p >= t) & (y == 0)).sum()
from_theory = from_validation = 0
for run in RUNS:
    pv, yv = sigmoid(run["L2 + keywords"]["val"][0]), run["L2 + keywords"]["val"][1]
    pt, yt = sigmoid(run["L2 + keywords"]["test"][0]), run["L2 + keywords"]["test"][1]
    t_val = grid[int(np.argmin([cost(pv, yv, t) for t in grid]))]
    from_validation += cost(pt, yt, t_val)
    from_theory += cost(pt, yt, 1 / 3)
in_hindsight = min(cost(sigmoid(z), y, t) for t in grid)
print(f"test cost at 0.5: {cost(sigmoid(z), y, 0.5)};  theory's 1/3: {from_theory};"
      f"  tuned on validation: {from_validation};  tuned on test itself: {in_hindsight}")

In [ ]:
%run measure.py

The post printed:

```text
split 0 test: TP 6  FP 2  FN 4  TN 28
no penalty     acc 0.833  precision 0.795  recall 0.445  F1 0.571  ROC-AUC 0.861  AP 0.714
L2             acc 0.809  precision 0.776  recall 0.330  F1 0.463  ROC-AUC 0.850  AP 0.688
L2 + keywords  acc 0.885  precision 0.891  recall 0.615  F1 0.728  ROC-AUC 0.888  AP 0.790
threshold 0.1: precision 0.563  recall 0.830  cost 197
threshold 0.2: precision 0.750  recall 0.750  cost 150
threshold 0.3: precision 0.789  recall 0.710  cost 154
threshold 0.4: precision 0.830  recall 0.660  cost 163
threshold 0.5: precision 0.891  recall 0.615  cost 169
threshold 0.6: precision 0.918  recall 0.560  cost 186
threshold 0.7: precision 0.934  recall 0.495  cost 209
threshold 0.8: precision 0.931  recall 0.405  cost 244
test cost at 0.5: 169;  theory's 1/3: 156;  tuned on validation: 167;  tuned on test itself: 149
```

## Can we trust a probability of 0.7?

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/measuring-a-classifier/#can-we-trust-a-probability-of-07)

In [ ]:
import numpy as np
from overfit import sigmoid, log_loss
from router_models import RUNS, pooled

def reliability(p, y, bins=5):
    """Mean predicted vs observed rate per probability bin, and the ECE."""
    edges = np.linspace(0, 1, bins + 1)
    which = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    rows, ece = [], 0.0
    for k in range(bins):
        m = which == k
        if m.any():
            rows.append((edges[k], edges[k + 1], m.sum(), p[m].mean(), y[m].mean()))
            ece += m.mean() * abs(p[m].mean() - y[m].mean())
    return rows, ece

for name in ["no penalty", "L2 + keywords"]:
    z, y = pooled(name, "test")
    rows, ece = reliability(sigmoid(z), y)
    print(f"{name}: ECE {ece:.3f}, Brier {((sigmoid(z) - y) ** 2).mean():.3f}")
    for lo, hi, n, conf, rate in rows:
        print(f"   p in [{lo:.1f}, {hi:.1f}): {n:3d} predictions, "
              f"mean p {conf:.2f}, actually serving {rate:.2f}")

# How big is ECE for probabilities that are calibrated by construction?
p = sigmoid(pooled("L2 + keywords", "test")[0])
rng = np.random.default_rng(0)
floor = [reliability(p, (rng.random(len(p)) < p).astype(float))[1] for _ in range(500)]
print(f"ECE of perfectly calibrated labels: mean {np.mean(floor):.3f},"
      f" 95th percentile {np.percentile(floor, 95):.3f}")

# Temperature scaling: one number tau, fit on each split's validation set.
grid = np.exp(np.linspace(np.log(0.2), np.log(5), 400))
for name in ["no penalty", "L2", "L2 + keywords"]:
    before, after, taus = [], [], []
    for run in RUNS:
        zv, yv = run[name]["val"]
        tau = grid[np.argmin([log_loss(sigmoid(zv / t), yv) for t in grid])]
        zt, yt = run[name]["test"]
        before.append(sigmoid(zt)); after.append(sigmoid(zt / tau)); taus.append(tau)
    y = pooled(name, "test")[1]
    e0, e1 = reliability(np.concatenate(before), y)[1], reliability(np.concatenate(after), y)[1]
    print(f"{name:14s} tau median {np.median(taus):.2f} (range {min(taus):.2f}-{max(taus):.2f}):"
          f"  ECE {e0:.3f} -> {e1:.3f}")

The post printed:

```text
no penalty: ECE 0.109, Brier 0.134
   p in [0.0, 0.2): 632 predictions, mean p 0.03, actually serving 0.12
   p in [0.2, 0.4):  40 predictions, mean p 0.28, actually serving 0.62
   p in [0.4, 0.6):  34 predictions, mean p 0.50, actually serving 0.79
   p in [0.6, 0.8):  33 predictions, mean p 0.72, actually serving 0.64
   p in [0.8, 1.0):  61 predictions, mean p 0.92, actually serving 0.84
L2 + keywords: ECE 0.037, Brier 0.094
   p in [0.0, 0.2): 600 predictions, mean p 0.06, actually serving 0.08
   p in [0.2, 0.4):  41 predictions, mean p 0.31, actually serving 0.44
   p in [0.4, 0.6):  37 predictions, mean p 0.49, actually serving 0.54
   p in [0.6, 0.8):  35 predictions, mean p 0.71, actually serving 0.89
   p in [0.8, 1.0):  87 predictions, mean p 0.94, actually serving 0.93
ECE of perfectly calibrated labels: mean 0.016, 95th percentile 0.027
no penalty     tau median 1.99 (range 1.03-3.08):  ECE 0.109 -> 0.031
L2             tau median 1.14 (range 0.73-1.67):  ECE 0.061 -> 0.043
L2 + keywords  tau median 1.24 (range 0.28-2.03):  ECE 0.037 -> 0.027
```

## The test set you didn't have

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/measuring-a-classifier/#the-test-set-you-didnt-have)

In [ ]:
import numpy as np
from overfit import sigmoid
from router_data import split, featurize, tokens, vietnamese
from router_models import RUNS, pooled

# How much of each Vietnamese question does the English vocabulary see?
(train_set, *_), vocab = featurize(*split(seed=0))
z, y = RUNS[0]["L2 + keywords"]["vi"]
for r, p in zip(vietnamese, sigmoid(z)):
    if r["route"] == "serving":
        seen = [t for t in tokens(r["question"]) if t in vocab]
        print(f"p={p:.2f} {'caught' if p >= 0.5 else 'MISSED'}  seen={seen}  {r['question']}")

# Over all 20 splits: how often is each serving question caught, and how
# often is a non-serving question wrongly sent to serving?
for name in ["no penalty", "L2", "L2 + keywords"]:
    caught = np.mean([sigmoid(run[name]["vi"][0]) >= 0.5 for run in RUNS], axis=0)
    z, y = pooled(name, "vi")
    recall = ((sigmoid(z) >= 0.5) & (y == 1)).sum() / (y == 1).sum()
    false_alarm = ((sigmoid(z) >= 0.5) & (y == 0)).sum() / (y == 0).sum()
    print(f"{name:14s} recall {recall:.3f}  false-alarm rate {false_alarm:.3f}"
          f"  per-question catch rate: {' '.join(f'{c:.2f}' for c in caught[:10])}")

The post printed:

```text
p=0.94 caught  seen=['kv', 'cache', 'a', '70b']  KV cache của mô hình 70B lớn cỡ nào?
p=0.98 caught  seen=['i', 'decode', 'prefill', 'i', 'token']  Tại sao decode chậm hơn prefill trên mỗi token?
p=0.07 MISSED  seen=['i', 'd']  Làm sao phục vụ một mô hình cho nhiều người dùng cùng lúc?
p=0.17 MISSED  seen=['token']  Vì sao token đầu tiên ra chậm như vậy?
p=0.08 MISSED  seen=['a']  Lượng tử hóa ảnh hưởng thế nào đến độ trễ?
p=0.39 MISSED  seen=['gpu', 'i']  Một GPU H100 chạy được bao nhiêu yêu cầu đồng thời?
p=0.59 caught  seen=['continuous', 'batching']  Continuous batching tăng thông lượng ra sao?
p=0.89 caught  seen=['vllm', 'd', 'paged', 'attention']  vLLM có dùng paged attention không?
p=0.07 MISSED  seen=['i', 'i', 'i', 'd']  Tại sao máy chủ chậm lại khi có thêm người dùng?
p=0.31 MISSED  seen=['speculative', 'decoding', 'i']  Speculative decoding tiết kiệm được bao nhiêu thời gian?
no penalty     recall 0.735  false-alarm rate 0.065  per-question catch rate: 0.85 1.00 0.50 1.00 0.00 0.95 0.75 0.80 0.50 1.00
L2             recall 0.505  false-alarm rate 0.000  per-question catch rate: 0.65 1.00 0.05 0.65 0.00 0.80 0.55 0.55 0.05 0.75
L2 + keywords  recall 0.540  false-alarm rate 0.000  per-question catch rate: 1.00 1.00 0.00 0.00 0.00 0.95 1.00 1.00 0.00 0.45
```